### Apertura Segura con try...except:

* Escribe unha función chamada `abrir_ficheiro_seguro(nome_ficheiro, modo)` que intente abrir un ficheiro no modo especificado.
* Debe usar un bloque `try...except` para capturar calquera excepción (como `FileNotFoundError`).
* Se a apertura é exitosa, a función debe devolver o obxecto do ficheiro. Se hai un erro, debe amosar a mensaxe de erro e devolver `None`.
* Fai probas invocando a función cun ficheiro que non exista e despois cun que si exista (por exemplo, en modo lectura `"r"`). Non esquezas pechar o ficheiro se se abriu correctamente.

In [ ]:
def abrir_ficheiro_seguro(nome_ficheiro, modo):
    try:
        f = open(nome_ficheiro, modo)
        return f
    except Exception as e:
        print(f"Erro: {e}")
        return None

f = abrir_ficheiro_seguro("non_existe.txt", "r")
print(f)

with open("proba_apertura.txt", "w") as tmp:
    tmp.write("contido de proba")

f = abrir_ficheiro_seguro("proba_apertura.txt", "r")
if f:
    print(f.read())
    f.close()

---

### Diario de Notas Persoal

* Crea un programa que permita ao usuario escribir anotacións nun ficheiro chamado `diario.txt`.
* O programa debe:
    1. Preguntar ao usuario se quere (1) Ler todas as anotacións ou (2) Engadir unha nova anotación.
    2. Se escolle ler, debe abrir o ficheiro en modo lectura (`"r"`) e amosar todo o seu contido na pantalla. Controla o erro se o ficheiro aínda non existe.
    3. Se escolle engadir, debe pedirlle ao usuario unha liña de texto e **engadila** ao final do ficheiro (modo `"a"`), incluíndo a data e hora actual ao comezo da liña (usa o módulo `datetime`).


In [ ]:
from datetime import datetime

def diario():
    while True:
        print("\n1. Ler anotacións\n2. Engadir anotación\n3. Saír")
        opcion = input("Escolle unha opción: ").strip()

        if opcion == "1":
            try:
                with open("diario.txt", "r") as f:
                    contido = f.read()
                    print(contido if contido else "O diario está baleiro.")
            except FileNotFoundError:
                print("O ficheiro diario.txt aínda non existe.")

        elif opcion == "2":
            nota = input("Escribe a túa anotación: ")
            with open("diario.txt", "a") as f:
                f.write(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {nota}\n")
            print("Anotación gardada.")

        elif opcion == "3":
            break

        else:
            print("Opción non válida.")

diario()

---

### Analizador de Ficheiro de Texto

* Pídelle ao usuario o nome dun ficheiro de texto.
* O programa debe abrir o ficheiro e calcular:
    * Número total de liñas.
    * Número total de palabras.
    * Número total de caracteres.
* Crea un ficheiro `proba.txt` con algunhas liñas de texto e comproba que os cálculos son correctos.

In [ ]:
with open("proba.txt", "w") as f:
    f.write("Ola mundo\nEste é un ficheiro de proba\nTen tres liñas de texto")

nome = input("Introduce o nome do ficheiro: ")

try:
    with open(nome, "r") as f:
        contido = f.read()
    lineas = contido.splitlines()
    palabras = contido.split()
    print(f"Liñas: {len(lineas)}")
    print(f"Palabras: {len(palabras)}")
    print(f"Caracteres: {len(contido)}")
except FileNotFoundError:
    print("Ficheiro non atopado.")

---

### Copiador de Ficheiros con Número de Liña

* Pídelle ao usuario o nome dun ficheiro de orixe.
* Crea un novo ficheiro denominado `copia_con_numeros.txt`.
* Le o ficheiro de orixe **liña por liña** e escribe cada liña no novo ficheiro, pero **engadíndolle o número de liña ao principio**.
* Por exemplo, se unha liña orixinal é "Hola mundo", no novo ficheiro debe aparecer "1 Hola mundo".

In [ ]:
nome_orixe = input("Introduce o nome do ficheiro de orixe: ")

try:
    with open(nome_orixe, "r") as entrada, open("copia_con_numeros.txt", "w") as saida:
        for num, linea in enumerate(entrada, start=1):
            saida.write(f"{num} {linea}")
    print("Copia creada en copia_con_numeros.txt")
except FileNotFoundError:
    print("Ficheiro de orixe non atopado.")

---

### Estracto bancario en CSV

* Crea un programa para ler e almacenar un estracto bancario do teu banco en CSV.

In [ ]:
import csv
import chardet
from datetime import datetime

movementos = []

def detectar_encoding(ruta):
    with open(ruta, "rb") as f:
        resultado = chardet.detect(f.read())
    return resultado["encoding"]

def cargar_csv():
    global movementos
    ruta = input("Ruta do ficheiro CSV: ")
    try:
        enc = detectar_encoding(ruta)
        with open(ruta, "r", encoding=enc) as f:
            reader = csv.DictReader(f, delimiter=";")
            movementos = [fila for fila in reader]
        print(f"{len(movementos)} movementos cargados. Cabeceiras: {list(movementos[0].keys()) if movementos else 'N/A'}")
    except FileNotFoundError:
        print("Ficheiro non atopado.")

def buscar_concepto():
    if not movementos:
        print("Carga primeiro os datos.")
        return
    termo = input("Concepto a buscar: ").lower()
    chave = next((c for c in movementos[0].keys() if "concepto" in c.lower()), None)
    if not chave:
        print("Non se atopou a columna de concepto.")
        return
    resultados = [m for m in movementos if termo in str(m.get(chave, "")).lower()]
    for m in resultados:
        print(m)
    print(f"Total: {len(resultados)} resultados.")

def buscar_cantidade():
    if not movementos:
        print("Carga primeiro os datos.")
        return
    cantidade = input("Cantidade a buscar: ")
    chave = next((c for c in movementos[0].keys() if "importe" in c.lower()), None)
    if not chave:
        print("Non se atopou a columna de importe.")
        return
    resultados = [m for m in movementos if str(m.get(chave, "")).replace(",", ".") == cantidade.replace(",", ".")]
    for m in resultados:
        print(m)
    print(f"Total: {len(resultados)} resultados.")

def suma_por_concepto():
    if not movementos:
        print("Carga primeiro os datos.")
        return
    termo = input("Concepto para sumar: ").lower()
    chave_c = next((c for c in movementos[0].keys() if "concepto" in c.lower()), None)
    chave_i = next((c for c in movementos[0].keys() if "importe" in c.lower()), None)
    if not chave_c or not chave_i:
        print("Non se atoparon as columnas necesarias.")
        return
    total = 0.0
    for m in movementos:
        if termo in str(m.get(chave_c, "")).lower():
            try:
                total += float(str(m.get(chave_i, "0")).replace(",", "."))
            except ValueError:
                pass
    print(f"Suma para '{termo}': {total:.2f}")

def buscar_entre_datas():
    if not movementos:
        print("Carga primeiro os datos.")
        return
    chave_d = next((c for c in movementos[0].keys() if "data" in c.lower()), None)
    if not chave_d:
        print("Non se atopou a columna de data.")
        return
    inicio_str = input("Data de inicio (DD-MM-YYYY): ")
    fin_str = input("Data de fin (DD-MM-YYYY): ")
    try:
        inicio = datetime.strptime(inicio_str, "%d-%m-%Y")
        fin = datetime.strptime(fin_str, "%d-%m-%Y")
    except ValueError:
        print("Formato de data incorrecto.")
        return
    for m in movementos:
        try:
            data = datetime.strptime(str(m.get(chave_d, "")), "%d-%m-%Y")
            if inicio <= data <= fin:
                print(m)
        except ValueError:
            pass

def menu_banco():
    while True:
        print("\n1. Cargar CSV\n2. Buscar por concepto\n3. Buscar por cantidade\n4. Suma por concepto\n5. Movementos entre datas\n6. Saír")
        op = input("Opción: ").strip()
        if op == "1":
            cargar_csv()
        elif op == "2":
            buscar_concepto()
        elif op == "3":
            buscar_cantidade()
        elif op == "4":
            suma_por_concepto()
        elif op == "5":
            buscar_entre_datas()
        elif op == "6":
            break
        else:
            print("Opción non válida.")

menu_banco()

---

### Pandas 

Podes tratar de converter o exercicio anterior usando Pandas.

In [ ]:
import pandas as pd
import chardet
from datetime import datetime

df = None

def detectar_encoding(ruta):
    with open(ruta, "rb") as f:
        return chardet.detect(f.read())["encoding"]

def cargar_csv_pandas():
    global df
    ruta = input("Ruta do ficheiro CSV: ")
    try:
        enc = detectar_encoding(ruta)
        df = pd.read_csv(ruta, sep=";", encoding=enc, decimal=",")
        print(f"Cargadas {len(df)} filas. Columnas: {list(df.columns)}")
    except FileNotFoundError:
        print("Ficheiro non atopado.")

def buscar_concepto_pd():
    if df is None:
        print("Carga primeiro os datos.")
        return
    col = next((c for c in df.columns if "concepto" in c.lower()), None)
    if not col:
        print("Non se atopou a columna de concepto.")
        return
    termo = input("Concepto a buscar: ").lower()
    resultado = df[df[col].str.lower().str.contains(termo, na=False)]
    print(resultado.to_string())
    print(f"Total: {len(resultado)} resultados.")

def buscar_cantidade_pd():
    if df is None:
        print("Carga primeiro os datos.")
        return
    col = next((c for c in df.columns if "importe" in c.lower()), None)
    if not col:
        print("Non se atopou a columna de importe.")
        return
    try:
        valor = float(input("Cantidade a buscar: ").replace(",", "."))
        resultado = df[df[col] == valor]
        print(resultado.to_string())
        print(f"Total: {len(resultado)} resultados.")
    except ValueError:
        print("Cantidade non válida.")

def suma_concepto_pd():
    if df is None:
        print("Carga primeiro os datos.")
        return
    col_c = next((c for c in df.columns if "concepto" in c.lower()), None)
    col_i = next((c for c in df.columns if "importe" in c.lower()), None)
    if not col_c or not col_i:
        print("Non se atoparon as columnas necesarias.")
        return
    termo = input("Concepto para sumar: ").lower()
    filtro = df[df[col_c].str.lower().str.contains(termo, na=False)]
    print(f"Suma: {filtro[col_i].sum():.2f}")

def datas_pd():
    if df is None:
        print("Carga primeiro os datos.")
        return
    col_d = next((c for c in df.columns if "data" in c.lower()), None)
    if not col_d:
        print("Non se atopou a columna de data.")
        return
    inicio_str = input("Data de inicio (DD-MM-YYYY): ")
    fin_str = input("Data de fin (DD-MM-YYYY): ")
    try:
        datas = pd.to_datetime(df[col_d], format="%d-%m-%Y", errors="coerce")
        inicio = pd.to_datetime(inicio_str, format="%d-%m-%Y")
        fin = pd.to_datetime(fin_str, format="%d-%m-%Y")
        resultado = df[(datas >= inicio) & (datas <= fin)]
        print(resultado.to_string())
    except Exception as e:
        print(f"Erro: {e}")

def menu_pandas():
    while True:
        print("\n1. Cargar CSV\n2. Buscar por concepto\n3. Buscar por cantidade\n4. Suma por concepto\n5. Movementos entre datas\n6. Saír")
        op = input("Opción: ").strip()
        if op == "1":
            cargar_csv_pandas()
        elif op == "2":
            buscar_concepto_pd()
        elif op == "3":
            buscar_cantidade_pd()
        elif op == "4":
            suma_concepto_pd()
        elif op == "5":
            datas_pd()
        elif op == "6":
            break
        else:
            print("Opción non válida.")

menu_pandas()

---

###  Xestor de Contactos en JSON

* Crea un programa para xestionar unha axenda de contactos nun ficheiro de texto `contactos.JSON`.

In [ ]:
import json
import os

FICHEIRO = "contactos.JSON"

def cargar_contactos():
    if not os.path.exists(FICHEIRO):
        return []
    with open(FICHEIRO, "r") as f:
        return json.load(f)

def gardar_contactos(contactos):
    with open(FICHEIRO, "w") as f:
        json.dump(contactos, f, indent=2, ensure_ascii=False)

def engadir_contacto():
    contactos = cargar_contactos()
    nome = input("Nome: ")
    telefono = input("Teléfono: ")
    email = input("Email: ")
    contactos.append({"nome": nome, "telefono": telefono, "email": email})
    gardar_contactos(contactos)
    print("Contacto engadido.")

def buscar_contacto():
    contactos = cargar_contactos()
    nome = input("Nome a buscar: ").lower()
    resultados = [c for c in contactos if nome in c["nome"].lower()]
    if resultados:
        for c in resultados:
            print(f"Nome: {c['nome']} | Teléfono: {c['telefono']} | Email: {c['email']}")
    else:
        print("Non se atoparon contactos.")

def listar_contactos():
    contactos = cargar_contactos()
    if not contactos:
        print("Non hai contactos gardados.")
        return
    for i, c in enumerate(contactos, start=1):
        print(f"{i}. {c['nome']} | {c['telefono']} | {c['email']}")

def menu_contactos():
    while True:
        print("\n1. Engadir contacto\n2. Buscar contacto\n3. Listar todos\n4. Saír")
        op = input("Opción: ").strip()
        if op == "1":
            engadir_contacto()
        elif op == "2":
            buscar_contacto()
        elif op == "3":
            listar_contactos()
        elif op == "4":
            break
        else:
            print("Opción non válida.")

menu_contactos()

---

### Cálculo con Datos Numéricos usando NumPy

In [ ]:
import numpy as np

with open("matriz.txt", "w") as f:
    f.write("1.5 2.3 4.8\n5.1 3.2 7.4\n9.6 8.7 6.5\n")

matriz = np.loadtxt("matriz.txt")

print(f"Media de todos os números: {matriz.mean():.2f}")
print(f"Suma de cada fila: {matriz.sum(axis=1)}")
print(f"Valor máximo de cada columna: {matriz.max(axis=0)}")

np.savetxt("matriz_dobre.txt", matriz * 2, fmt="%.2f")
print("Ficheiro matriz_dobre.txt gardado.")